# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [4]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided context, the most common issues with loans involve problems related to loan servicing and management. Specifically, these issues include:\n\n- Errors in loan balances and misapplication of payments\n- Receiving bad or incorrect information about the loan\n- Difficulty with payment handling, including restrictions on how additional funds are applied\n- Discrepancies and inaccuracies on credit reports and account statuses\n- Problems with repayments, forbearance, and loan transfers without proper notifications\n- Issues with loan balances growing unexpectedly or not matching documentation\n- Mishandling of private or sensitive information and privacy violations\n- Challenges in obtaining loan forgiveness or discharge\n\nOverall, a prominent and recurring problem is mismanagement and inaccuracies in loan handling, especially errors in balances, payment application, and lack of transparency from loan servicers.\n\nIf you have a specific aspect you're interested in, p

In [11]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, some complaints indicate that they were not handled in a timely manner. Specifically:\n\n- One complaint from MOHELA (Complaint ID: 12709087) was marked as "Timely response?": No, and it involved delays in response and unresolved issues after multiple contacts over several weeks.\n- Another from Maximus Federal Services (Complaint ID: 12975634) was marked as "Timely response?": Yes, but the complaint details suggest ongoing delays and unresolved issues over months.\n- EdFinancial Services complaints (Complaint IDs: 12973003, 12744910, 13056764) show responses marked as "Timely response?": Yes, but the narratives describe unresolved issues persisting over weeks to months, indicating delays in resolution despite official response times.\n\nOverall, at least some complaints experienced delays and were not handled promptly, with delays ranging from days to over a year in unresolved cases.'

In [12]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People have failed to pay back their loans for several interconnected reasons, including:\n\n1. **Accumulation of interest and ongoing debt increasing despite payments:** Borrowers often find that interest continues to accrue, especially when loans are put into forbearance or deferment, which can negate any early payments and extend the repayment period. As a result, the total debt can grow over time, making it difficult or seemingly impossible to pay off.\n\n2. **Unmanageable repayment options and financial hardship:** Many borrowers cannot afford higher monthly payments or additional payments toward principal without sacrificing essential living expenses. Income stagnation, stagnant wages, and rising living costs make increasing payments impractical.\n\n3. **Lack of clear or transparent communication from servicers:** Borrowers report being inadequately informed about the status of their loans, including when payments are due, changes in loan servicers, or delinquency notices. This 

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be dealing with the lender or servicer, particularly related to misinformation, poor communication, and unfair repayment practices. Common problems include disputes over fees, difficulty applying payments correctly, challenges in obtaining accurate loan information, and issues with how payments are applied (such as excessive interest or inability to pay down principal).'

In [16]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, all the complaints listed were responded to with a "Closed with explanation" status and marked as "Yes" for being handled in a timely manner. Therefore, it appears that any complaints mentioned did get handled in a timely manner.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including issues with their loan servicers such as being steered into incorrect types of payment plans or forbearances, inadequate communication from the lenders or servicers, technical errors like payments being reversed or not processed properly, and lack of timely updates or notices about changes to their accounts or payment status. In some cases, borrowers were unaware of the transfer of their loans to different companies, did not receive important notifications, or experienced administrative problems that resulted in overdue accounts and negative impacts on their credit scores.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

## Answer: 

My example: What was the complaint listed in Complaint ID: 123456? Was it something related to timeliness?

I would say that BM25 would be better than embeddings with this query because we want information regarding a very certain complaint ID so the retriever would need to search for that exact ID. BM25 works better for this since it would search for information concerning that specific complaint ID (text similarity), but embeddings finds semantically similar information, so it would likely find irrelevant information about complaints or timeliness in general.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided information, the most common issues with loans involve problems related to loan servicing, such as errors in loan balances, misapplied payments, wrongful denials of payment plans, and mishandling of loan data. Many complaints also mention incorrect or conflicting information about loan balances, unauthorized transfers of loans, lack of communication or documentation, and potential violations of privacy laws. Overall, a significant and recurring issue appears to be the mishandling and miscommunication by loan servicers.'

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, at least one complaint explicitly indicates that it was not handled in a timely manner. Specifically, the complaint regarding the student loan issue with Maximus Federal Services, Inc. states that the matter has been open for nearly 18 months with no resolution, and the complainant is still awaiting a response and resolution despite multiple requests over time. \n\nAdditionally, another complaint about EdFinancial Services mentions a problem that has persisted for over 2-3 weeks, and the issue has not been resolved yet. The complaint indicates ongoing difficulties despite multiple follow-ups.\n\nTherefore, yes, some complaints did not get handled in a timely manner.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans primarily due to a lack of clear and consistent information about the repayment process, unexpected transfer of loans without proper notification, difficulties with account access, and the accumulation of interest that made repayment challenging. Additionally, some borrowers were unaware that they were responsible for repayment, as they believed they were not required to pay or did not fully understand the terms and interest implications. The complexity of loan management, combined with inadequate communication from lenders and servicers, contributed to borrowers' inability to successfully repay their loans."

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [23]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [24]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided data, the most common issues with loans tend to revolve around mismanagement and lack of transparency. Specifically, frequent issues include:\n\n- Errors or discrepancies in loan balances and interest calculations\n- Inadequate or confusing communication about loan terms, changes, or payment status\n- Problems with loan transfer or servicing changes without proper notification\n- Incorrect reporting of account status, such as delinquencies or defaults\n- Challenges in obtaining accurate information or validation about the loan\n- Difficulties in managing or applying payments correctly, often leading to unwanted interest accumulation or increased balances\n- Mishandling associated with deferment, forbearance, or consolidation procedures\n\nOverall, a primary concern appears to be that borrowers face confusion, errors, and lack of transparency, which complicate repayment and negatively impact credit reports.\n\nTherefore, the most common issue with loans, as reflec

In [26]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, some complaints indicate that they were not handled in a timely manner. Specifically:\n\n- One complaint (row 441) from a consumer regarding a student loan issue was marked as "Timely response?": "No," indicating it was not handled promptly.\n- Another complaint (row 400) also from a student loan borrower was marked "Yes" for timely response, so it was handled promptly.\n- Several other complaints (rows 67, 95, 236, 474, 640, 487, 95, 238, 611, 541, 503, 610, etc.) mention delays, lack of resolution over extended periods (some over a year), or violations of law regarding response times and resolution.\n\nIn particular, multiple complaints highlight that the complaints or issues were not addressed within appropriate or promised timeframes, with some cases reporting no response for many months or over a year.\n\nTherefore, the answer is: Yes, some complaints did not get handled in a timely manner.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of systemic issues, miscommunication, and unfavorable repayment options. Many borrowers were misled or lacked sufficient information about available repayment plans, such as income-driven repayment or loan rehabilitation, which could have made payments more manageable. Instead, they were often steered into forbearance or other strategies that caused interest to accrue rapidly, increasing their total debt over time. \n\nAdditionally, borrowers faced challenges like:\n\n- Lack of proper notices or communication from servicers, leading to unawareness of when repayment resumed or changes in account status.\n- Errors or delays in reporting loan statuses, which negatively impacted credit scores and caused financial hardship.\n- Servicing practices that sometimes involved improper handling, coercive tactics, or not providing legal alternatives, such as IDR options.\n- Transfer of loans between different servicers without pr

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

## Answer: 

Creating multiple versions of the same query helps the AI system find more answers because the slight differences in words/phrases in the rephrased questions might bring out more information that is relevant but was not retrieved before. This is because some documents themselves might use different words/phrases so, in a sense, you are covering all bases to find all the relevant information for a single given query.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [28]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [29]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [30]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [31]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [32]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [33]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be problems related to the servicing of student loans. These include errors in loan balances, misapplied payments, wrongful denials of payment plans, and misconduct by loan servicers. Additionally, issues such as incorrect or inconsistent reporting of account status, disputes over interest rates and fees, and problems caused by the sale or transfer of loans are prominent.'

In [34]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, yes, there were complaints that did not get handled in a timely manner. Specifically, the complaints with IDs 12709087 and 12935889 from Mohela, received on 03/28/25 and 04/11/25 respectively, both have a response status indicating they were "No" for being timely. This suggests these complaints were not addressed within the expected timeframe.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People often fail to pay back their loans due to a variety of factors, including financial hardship, mismanagement, and lack of proper information or communication from lenders. In the provided context, some specific reasons include:\n\n- **Financial hardship and inability to secure employment or generate enough income to make payments**, as seen in the case of students who experienced severe financial difficulties after graduating and relied on deferment or forbearance.\n- **Misinformation or lack of transparency from loan servicers or educational institutions**, such as not properly informing borrowers about payment start dates, grace periods, or the consequences of default.\n- **Long-term financial consequences of student loans were not clearly communicated**, leading borrowers to underestimate the difficulty of repayment.\n- **Excessive debt burden and interest accumulation**, especially when deferments and forbearances are used, can make repayment more challenging.\n- **Administr

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [36]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [37]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [38]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issues with loans include:\n\n- Dealing with lenders or servicers, including receiving bad information, misapplication of payments, and wrongful denials of payment plans.\n- Trouble with how payments are handled, such as inability to apply additional funds to the principal or pay off smaller loans more quickly.\n- Incorrect or misleading information on credit reports, including inaccurate loan balances, reporting of delinquencies or default statuses without proper notice, and errors in account status.\n- Unjustified increases in interest rates, fees, or balances, often after loans are sold or transferred without proper notification.\n- Fraud concerns, such as unauthorized transfers, identity confusion, or inadequate validation of loan ownership.\n- Problems related to loan forgiveness, cancellation, or discharge, especially following administrative transfers or legal changes.\n- Lack of communication or transparency from loan servicers, r

In [39]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints data, yes, there are complaints indicating that some complaints were not handled in a timely manner. Specifically, at least two complaints explicitly state a failure to respond within the expected timeframe:\n\n- Complaint ID: 12739706 (submitted to MOHELA in NJ) received on 04/01/25, titled "Dealing with your lender or servicer," which was marked as "No" in the "Timely response" field, meaning it was not responded to in time.\n\n- Complaint ID: 12935889 (submitted to MOHELA in CO) received on 04/11/25, also marked as "No" for timely response, indicating it was not handled in a timely manner.\n\nAdditionally, many complaints mention delays, poor follow-up, or no response at all, which suggests that some complaints were not handled promptly. \n\nIn summary, yes, there were complaints that did not get handled in a timely manner as per the information available.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily because of several systemic and mismanagement issues, including:\n\n1. **Lack of Proper Communication:** Many borrowers were not adequately notified about their loan status, payment due dates, or changes in servicing companies. For example, borrowers reported not receiving notifications when their loans were transferred or when repayment was supposed to resume, leading to unintentional delinquency.\n\n2. **Interest Accumulation and Misunderstanding of Loan Terms:** Borrowers often were not informed about how interest would accrue during forbearance or deferment, leading to balances ballooning over time. Many felt misled about payment options, especially regarding income-driven repayment plans and loan forgiveness programs.\n\n3. **Inappropriate Servicer Practices (e.g., Forbearance Steering):** Some borrowers experienced "forbearance steering," where they were repeatedly placed into long-term forbearances instead of being informed of or 

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [41]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [42]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [43]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [44]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [45]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [46]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints data, the most common issue with loans appears to be related to problems with loan servicing and administration. Common sub-issues include:\n\n- Struggling to repay loans, often due to delays or problems with forgiveness or discharge processes.\n- Issues with how payments are handled, such as incorrect amounts or auto-debit failures.\n- Confusion or disputes over loan account status, default notices, or misreported delinquencies.\n- Problems with loan information reporting and verification, including incorrect or unauthorized reporting.\n- Lack of communication or clarity from loan servicers about account changes, balances, or payment plans.\n\nOverall, many complaints revolve around servicing issues—errors in payment processing, miscommunication about loan status or payment plans, and challenges with loan forgiveness or discharge processes.'

In [47]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, some complaints did not get handled in a timely manner. \n\nSpecifically, for the complaint regarding Nelnet, Inc. received on 05/04/25 (Complaint ID: 13331376), the company responded with "Closed with explanation," but the complaint indicates issues such as failed responses to letters and continued violations, which suggest ongoing unresolved concerns. The complaint details serious misconduct and a lack of response despite multiple letters sent, indicating it was not handled promptly or adequately.\n\nSimilarly, other complaints also show delays or lack of proper handling—such as the dispute about bad information about a loan and the one about trouble with payments being processed, where the company response was "Closed with explanation," implying the issues may not have been fully resolved to the consumer\'s satisfaction.\n\nTherefore, the answer is: **Yes, some complaints did not get handled in a timely manner.**'

In [48]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for several reasons reflected in the complaints:\n\n1. **Lack of transparency and poor communication**: Several complaints mention difficulty in obtaining clear information from lenders or servicers, delays, long waits on calls, and unhelpful responses, which can hinder borrowers' ability to understand their loan status or repayment options.\n\n2. **Administrative errors and mismanagement**: Some borrowers experienced issues like payments not being properly processed, misattribution of payments, or incorrect account statuses (e.g., being reported in default despite never being in default).\n\n3. **Problems with documentation and certification**: Borrowers attempting to qualify for forgiveness or discharge faced stalling, incomplete or incorrect documentation processes, and delays that made repayment or forgiveness challenging.\n\n4. **Disputes and legal issues**: Complaints include disputes over debt legitimacy, claims of illegal reporting, or the

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

## Answer:

If sentences are short and highly repetitive, semantic chunking might not perform well because it relies on finding semantically similar chunks, and if there are many text chunks that are relatively identical, there will either be a lot or very little chunks retrieved. I would adjust the algorithm by changing the chunking method to RecursiveCharacterTextSplitter or maybe chunking by section or paragraph.

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [51]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

In [57]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(loan_complaint_data[:20], testset_size=10)

dataset.to_pandas()

Applying SummaryExtractor:   0%|          | 0/14 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/20 [00:00<?, ?it/s]

Node 0645d7e9-5460-4bf3-8741-6e143334e40f does not have a summary. Skipping filtering.
Node 303fd1a6-fb9f-42e5-aa01-f314d22f82b2 does not have a summary. Skipping filtering.
Node 5a5c771e-7ea0-4c2d-b148-470c277cf8fd does not have a summary. Skipping filtering.
Node a549455a-7d5d-41c9-9e83-8c024bcce3a7 does not have a summary. Skipping filtering.
Node a1e38a51-49c9-4d5a-b037-85b216e75788 does not have a summary. Skipping filtering.
Node 807aab3a-2d77-485f-90a5-8f7e5dfc1a9b does not have a summary. Skipping filtering.


Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/54 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,When did the federal student loan COVID-19 for...,[The federal student loan COVID-19 forbearance...,The federal student loan COVID-19 forbearance ...,single_hop_specifc_query_synthesizer
1,How does the improper processing of my IDR/IBR...,[I submitted my annual Income-Driven Repayment...,"According to the provided context, Aidvantage ...",single_hop_specifc_query_synthesizer
2,Why my student loan info got out when FERPA su...,[My personal and financial data was compromise...,My personal and financial data was compromised...,single_hop_specifc_query_synthesizer
3,Why nelnet say my issuer is somewhere else whe...,"[According to Studentaid.gov, Im to get an ema...",Studentaid.gov says that your issuer is nelnet...,single_hop_specifc_query_synthesizer
4,Why is it so hard to get teacher loan forgiven...,[<1-hop>\n\nI have provided documentation rela...,It is hard to get teacher loan forgiveness bec...,multi_hop_abstract_query_synthesizer
5,How do unfair and deceptive practices relate t...,[<1-hop>\n\nIllegal Student Loan Reporting & C...,Unfair and deceptive practices are demonstrate...,multi_hop_abstract_query_synthesizer
6,"How have payment processing issues, such as de...",[<1-hop>\n\nThe federal student loan COVID-19 ...,Payment processing issues have significantly c...,multi_hop_abstract_query_synthesizer
7,how come aidvantage mess up my loan payment ca...,[<1-hop>\n\nI submitted my annual Income-Drive...,aidvantage made a mistake with your loan payme...,multi_hop_abstract_query_synthesizer
8,How do the actions of the loan servicer and Ne...,[<1-hop>\n\nIllegal Student Loan Reporting & C...,The actions of the loan servicer and Nelnet po...,multi_hop_specific_query_synthesizer
9,Why is nelnet sayin my issuer is somewhere els...,"[<1-hop>\n\nAccording to Studentaid.gov, Im to...","Nelnet says your issuer is somewhere else, eve...",multi_hop_specific_query_synthesizer


In [66]:
# for test_row in dataset:
#   response = naive_retrieval_chain.invoke({"question" : test_row.eval_sample.user_input}) ## input with given retriever
#   test_row.eval_sample.response = response["response"].content
#   test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

In [67]:
# dataset.samples[0].eval_sample.response

'The federal student loan COVID-19 forbearance program ended on XX/XX/XXXX.'

In [ ]:
# from ragas import EvaluationDataset

# evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())

In [69]:
# from ragas import evaluate
# from ragas.llms import LangchainLLMWrapper

# evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

In [70]:
# from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
# from ragas import evaluate, RunConfig

# custom_run_config = RunConfig(timeout=360)

# result = evaluate(
#     dataset=evaluation_dataset,
#     metrics=[LLMContextRecall(), ContextEntityRecall(), NoiseSensitivity()],
#     llm=evaluator_llm,
#     run_config=custom_run_config
# )
# result

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

Exception raised in Job[11]: TimeoutError()
Exception raised in Job[17]: TimeoutError()
Exception raised in Job[23]: TimeoutError()
Exception raised in Job[29]: TimeoutError()
Exception raised in Job[41]: TimeoutError()
Exception raised in Job[47]: TimeoutError()
Exception raised in Job[53]: TimeoutError()
Exception raised in Job[59]: TimeoutError()
Exception raised in Job[65]: TimeoutError()
Exception raised in Job[71]: TimeoutError()


{'context_recall': 0.9111, 'faithfulness': 0.9015, 'factual_correctness(mode=f1)': 0.6050, 'answer_relevancy': 0.6144, 'context_entity_recall': 0.4168, 'noise_sensitivity(mode=relevant)': 0.1111}

In [80]:
from ragas import EvaluationDataset
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, ContextEntityRecall, NoiseSensitivity
from ragas import RunConfig
import time


def evaluate_all_retrievers_with_metrics(synthetic_dataset):
    """Evaluate all retrievers using the same synthetic dataset and evaluator"""
    
    retrievers = {
        "naive": naive_retrieval_chain,
        "bm25": bm25_retrieval_chain,
        "multi_query": multi_query_retrieval_chain,
        "parent_document": parent_document_retrieval_chain,
        "compression": contextual_compression_retrieval_chain,
        "ensemble": ensemble_retrieval_chain
    }
    
    results = {}
    
    for name, retriever_chain in retrievers.items():
        print(f"Evaluating {name} retriever...")
        
        # Measure latency and cost
        latencies = []
        costs = []
        
        # Process each test case and collect metrics
        for test_row in synthetic_dataset:
            print("now measuring latency") # Measure latency
            start_time = time.time()
            response = retriever_chain.invoke({"question": test_row.eval_sample.user_input})
            end_time = time.time()
            latencies.append(end_time - start_time)
            
            print("now estimating cost") # Estimate cost
            response_text = response["response"].content
            estimated_tokens = len(response_text) / 4
            cost = estimated_tokens * (0.00015 / 1000)  # gpt-4o-mini cost
            costs.append(cost)
            
            # Update dataset for Ragas evaluation (same as your existing code)
            test_row.eval_sample.response = response_text
            test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
        
        # Calculate latency and cost metrics
        latency_metrics = {
            "avg_latency_seconds": sum(latencies) / len(latencies),
            "min_latency_seconds": min(latencies),
            "max_latency_seconds": max(latencies)
        }
        
        cost_metrics = {
            "avg_cost_usd": sum(costs) / len(costs),
            "total_cost_usd": sum(costs),
            "min_cost_usd": min(costs),
            "max_cost_usd": max(costs)
        }
        
        # Get Ragas metrics (using the same evaluator_llm)
        evaluation_dataset = EvaluationDataset.from_pandas(synthetic_dataset.to_pandas())
        evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
        custom_run_config = RunConfig(timeout=360)
        
        print("now in the evaluation phase")
        ragas_result = evaluate(
            dataset=evaluation_dataset,
            metrics=[LLMContextRecall(), ContextEntityRecall(), NoiseSensitivity()],
            llm=evaluator_llm,  # Same evaluator for all
            run_config=custom_run_config
        )
        
        results[name] = {
            "latency_metrics": latency_metrics,
            "cost_metrics": cost_metrics,
            "ragas_metrics": ragas_result
        }
    
    return results

In [83]:
evaluate_all_retrievers_with_metrics(dataset)

Evaluating naive retriever...
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now in the evaluation phase


Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

Exception raised in Job[4]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[10]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Exception raised in Job[20]: TimeoutError()
Exception raised in Job[23]: TimeoutError()
Exception raised in Job[26]: TimeoutError()
Exception raised in Job[35]: TimeoutError()


Evaluating bm25 retriever...
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now in the evaluation phase


Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

Evaluating ensemble retriever...
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now in the evaluation phase


Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

Exception raised in Job[4]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[8]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Exception raised in Job[17]: TimeoutError()
Exception raised in Job[20]: TimeoutError()
Exception raised in Job[22]: TimeoutError()
Exception raised in Job[23]: TimeoutError()
Exception raised in Job[26]: TimeoutError()
Exception raised in Job[28]: TimeoutError()
Exception raised in Job[29]: TimeoutError()
Exception raised in Job[32]: TimeoutError()
Exception raised in Job[35]: TimeoutError()


{'naive': {'latency_metrics': {'avg_latency_seconds': 5.4087908665339155,
   'min_latency_seconds': 1.8014369010925293,
   'max_latency_seconds': 8.364365100860596},
  'cost_metrics': {'avg_cost_usd': 7.5646875e-05,
   'total_cost_usd': 0.0009077625,
   'min_cost_usd': 2.5499999999999997e-06,
   'max_cost_usd': 0.00012941249999999998},
  'ragas_metrics': {'context_recall': 0.8944, 'context_entity_recall': 0.3942, 'noise_sensitivity(mode=relevant)': 0.2759}},
 'bm25': {'latency_metrics': {'avg_latency_seconds': 5.9958977699279785,
   'min_latency_seconds': 1.180063009262085,
   'max_latency_seconds': 14.708261966705322},
  'cost_metrics': {'avg_cost_usd': 7.7103125e-05,
   'total_cost_usd': 0.0009252374999999999,
   'min_cost_usd': 2.7749999999999997e-06,
   'max_cost_usd': 0.00013905},
  'ragas_metrics': {'context_recall': 0.6232, 'context_entity_recall': 0.3170, 'noise_sensitivity(mode=relevant)': 0.1831}},
 'multi_query': {'latency_metrics': {'avg_latency_seconds': 10.192157725493113

In [ ]:
# Print comparison
results_retrievers = {'naive': {'latency_metrics': {'avg_latency_seconds': 5.4087908665339155,
   'min_latency_seconds': 1.8014369010925293,
   'max_latency_seconds': 8.364365100860596},
  'cost_metrics': {'avg_cost_usd': 7.5646875e-05,
   'total_cost_usd': 0.0009077625,
   'min_cost_usd': 2.5499999999999997e-06,
   'max_cost_usd': 0.00012941249999999998},
  'ragas_metrics': {'context_recall': 0.8944, 'context_entity_recall': 0.3942, 'noise_sensitivity(mode=relevant)': 0.2759}},
 'bm25': {'latency_metrics': {'avg_latency_seconds': 5.9958977699279785,
   'min_latency_seconds': 1.180063009262085,
   'max_latency_seconds': 14.708261966705322},
  'cost_metrics': {'avg_cost_usd': 7.7103125e-05,
   'total_cost_usd': 0.0009252374999999999,
   'min_cost_usd': 2.7749999999999997e-06,
   'max_cost_usd': 0.00013905},
  'ragas_metrics': {'context_recall': 0.6232, 'context_entity_recall': 0.3170, 'noise_sensitivity(mode=relevant)': 0.1831}},
 'multi_query': {'latency_metrics': {'avg_latency_seconds': 10.192157725493113,
   'min_latency_seconds': 4.356123208999634,
   'max_latency_seconds': 16.005552291870117},
  'cost_metrics': {'avg_cost_usd': 7.1703125e-05,
   'total_cost_usd': 0.0008604375,
   'min_cost_usd': 2.7749999999999997e-06,
   'max_cost_usd': 0.0001264125},
  'ragas_metrics': {'context_recall': 0.9111, 'context_entity_recall': 0.3892, 'noise_sensitivity(mode=relevant)': 0.3333}},
 'parent_document': {'latency_metrics': {'avg_latency_seconds': 10.923909882704416,
   'min_latency_seconds': 2.113287925720215,
   'max_latency_seconds': 22.10703206062317},
  'cost_metrics': {'avg_cost_usd': 6.55125e-05,
   'total_cost_usd': 0.00078615,
   'min_cost_usd': 2.7749999999999997e-06,
   'max_cost_usd': 0.0001135125},
  'ragas_metrics': {'context_recall': 0.6847, 'context_entity_recall': 0.3724, 'noise_sensitivity(mode=relevant)': 0.2519}},
 'compression': {'latency_metrics': {'avg_latency_seconds': 8.2982826034228,
   'min_latency_seconds': 3.8291120529174805,
   'max_latency_seconds': 25.689271688461304},
  'cost_metrics': {'avg_cost_usd': 6.409375e-05,
   'total_cost_usd': 0.000769125,
   'min_cost_usd': 2.7749999999999997e-06,
   'max_cost_usd': 0.0001112625},
  'ragas_metrics': {'context_recall': 0.7804, 'context_entity_recall': 0.3729, 'noise_sensitivity(mode=relevant)': 0.2311}},
 'ensemble': {'latency_metrics': {'avg_latency_seconds': 22.435351471106213,
   'min_latency_seconds': 9.048939228057861,
   'max_latency_seconds': 34.87581014633179},
  'cost_metrics': {'avg_cost_usd': 8.5390625e-05,
   'total_cost_usd': 0.0010246875,
   'min_cost_usd': 2.7e-06,
   'max_cost_usd': 0.0001291875},
  'ragas_metrics': {'context_recall': 0.9444, 'context_entity_recall': 0.4103, 'noise_sensitivity(mode=relevant)': 0.0000}}}

print("Retriever Performance Comparison:")
print("=" * 60)

for name, result in results_retrievers.items():
    print(f"\n{name.upper()} RETRIEVER:")
    print(f"  Latency: {result['latency_metrics']['avg_latency_seconds']:.2f}s")
    print(f"  Cost: ${result['cost_metrics']['avg_cost_usd']:.4f}")
    print(f"  Context Recall: {result['ragas_metrics']['context_recall']:.3f}")
    print(f"  Context Entity Recall: {result['ragas_metrics']['context_entity_recall']:.3f}")
    print(f"  Noise Sensitivity: {result['ragas_metrics']['noise_sensitivity(mode=relevant)']:.3f}")

In [90]:
# Create a side-by-side comparison table
import pandas as pd

# Prepare data for DataFrame
comparison_data = []
for name, result in results_retrievers.items():
    comparison_data.append({
        'Retriever': name.upper(),
        'Avg Latency (s)': round(result['latency_metrics']['avg_latency_seconds'], 2),
        'Avg Cost ($)': round(result['cost_metrics']['avg_cost_usd'], 6),
        'Context Recall': round(result['ragas_metrics']['context_recall'], 3),
        'Context Entity Recall': round(result['ragas_metrics']['context_entity_recall'], 3),
        'Noise Sensitivity': round(result['ragas_metrics']['noise_sensitivity(mode=relevant)'], 3)
    })

# Create DataFrame and display
comparison_df = pd.DataFrame(comparison_data)
print("\n" + "="*80)
print("SIDE-BY-SIDE RETRIEVER COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))


SIDE-BY-SIDE RETRIEVER COMPARISON
      Retriever  Avg Latency (s)  Avg Cost ($)  Context Recall  Context Entity Recall  Noise Sensitivity
          NAIVE             5.41      0.000076           0.894                  0.394              0.276
           BM25             6.00      0.000077           0.623                  0.317              0.183
    MULTI_QUERY            10.19      0.000072           0.911                  0.389              0.333
PARENT_DOCUMENT            10.92      0.000066           0.685                  0.372              0.252
    COMPRESSION             8.30      0.000064           0.780                  0.373              0.231
       ENSEMBLE            22.44      0.000085           0.944                  0.410              0.000
